In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# 1. パラメータ設定
# rx1, rx2 を変数とし、ry (およびその他) を固定します
params = {
    'ry': [1.0, 1.0],               # [ry1, ry2] (固定)
    'lambda_diag': [1.0, 1.0],      # [l11, l22]
    'lambda_offdiag': [0.3, 0.3],   # [l12, l21]
    'c': [0.8, 0.6],                # [c1, c2]
    'd': [0.6, 0.8]                 # [d1, d2]
}

ry1, ry2 = params['ry']
l11, l22 = params['lambda_diag']
l12, l21 = params['lambda_offdiag']
c1, c2 = params['c']
d1, d2 = params['d']

# 2. メッシュグリッド (rx1, rx2 を変数とする)
res = 400
range_rx = 2.0
vals = np.linspace(0.001, range_rx, res)
RX1, RX2 = np.meshgrid(vals, vals)

# 3. 計算
# --- Det X & Det Y (すべて定数) ---
Det_x = l11 * l22 - l12 * l21
Det_y = c1 * d2 * l11 * l22 - c2 * d1 * l12 * l21

# --- Num X1 & X2 (すべて定数) ---
# Xの分子は ry に依存し、rx には依存しないため
Num_x1 = l22 * ry1 - l12 * ry2
Num_x2 = l11 * ry2 - l21 * ry1

# --- Num Y1 & Y2 (変数 RX1, RX2 に依存) ---
# Num_y1 = d2 * l22 * rx1 - d1 * l21 * rx2
Num_y1 = d2 * l22 * RX1 - d1 * l21 * RX2

# Num_y2 = c1 * l11 * rx2 - c2 * l12 * rx1
Num_y2 = c1 * l11 * RX2 - c2 * l12 * RX1

# 平衡点の値を計算
with np.errstate(divide='ignore', invalid='ignore'):
    # Detが定数なのでスカラー除算ですが、Num_yがグリッドなのでYはグリッドになります
    X1 = Num_x1 / Det_x  # 定数（スカラー）
    X2 = Num_x2 / Det_x  # 定数（スカラー）
    Y1 = Num_y1 / Det_y
    Y2 = Num_y2 / Det_y

# 4. 領域のカテゴリ分け
# X1, X2 はスカラーですが、broadcastingでグリッド比較に使えます
Category = np.zeros_like(Y1, dtype=int)

pos_x1 = X1 > 0
pos_x2 = X2 > 0
pos_y1 = Y1 > 0
pos_y2 = Y2 > 0

# カテゴリ番号定義:
# 1: 共存, 2: x1<0, 3: x2<0, 4: y1<0, 5: y2<0

# Yの死滅判定
Category[~pos_y1] = 4
Category[~pos_y2] = 5

# Xの死滅判定 (定数なので、条件を満たせば全画面が塗られます)
# 通常の研究パラメータ設定ではXは正と仮定されますが、念のため実装
if not pos_x1:
    mask_x1_die = np.ones_like(Category, dtype=bool) & pos_x2 # X2が生きてれば
    Category[mask_x1_die] = 2

if not pos_x2:
    mask_x2_die = np.ones_like(Category, dtype=bool) & pos_x1 # X1が生きてれば
    Category[mask_x2_die] = 3

# 共存領域
if pos_x1 and pos_x2:
    mask_coexist = pos_y1 & pos_y2
    Category[mask_coexist] = 1

# 5. プロット
fig, ax = plt.subplots(figsize=(10, 8))

# カラーマップ
colors = ['white', 'gold', 'skyblue', 'salmon', 'SlateBlue', 'IndianRed']
cmap = ListedColormap(colors)
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5], cmap.N)

im = ax.imshow(Category, extent=[0, range_rx, 0, range_rx], origin='lower',
               cmap=cmap, norm=norm, aspect='auto', alpha=0.9)

# --- 曲線 (境界線) 描画 ---
# 今回 Det関係の特異点はないので描画しません

x_line = np.linspace(0.001, range_rx, 500)

# 1. Num_y1 = 0 (y1の符号変化)
# d2 * l22 * rx1 = d1 * l21 * rx2
# rx2 = (d2 * l22 / (d1 * l21)) * rx1  -> 原点を通る直線
slope_y1 = (d2 * l22) / (d1 * l21)
y_line_num_y1 = slope_y1 * x_line
ax.plot(x_line, y_line_num_y1, color='SlateBlue', linestyle='-', linewidth=2, label=r'$y_1=0$ Boundary')

# 2. Num_y2 = 0 (y2の符号変化)
# c1 * l11 * rx2 = c2 * l12 * rx1
# rx2 = (c2 * l12 / (c1 * l11)) * rx1 -> 原点を通る直線
slope_y2 = (c2 * l12) / (c1 * l11)
y_line_num_y2 = slope_y2 * x_line
ax.plot(x_line, y_line_num_y2, color='IndianRed', linestyle='-', linewidth=2, label=r'$y_2=0$ Boundary')

# --- パラメータ表示 ---
textstr = '\n'.join(( 
    r'Fixed Parameters:',
    r'$r_y = [%.1f, %.1f]$' % tuple(params['ry']),
    r'$\lambda = [%.1f, %.1f, %.1f, %.1f]$' % (l11, l12, l21, l22),
    r'$c = [%.1f, %.1f]$' % (c1, c2),
    r'$d = [%.1f, %.1f]$' % (d1, d2)
))

props = dict(boxstyle='round', facecolor='white', alpha=0.9)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=props)

# 軸ラベルとタイトル
ax.set_xlim(0, range_rx)
ax.set_ylim(0, range_rx)
ax.set_xlabel(r'Parameter $r_{x1}$')
ax.set_ylabel(r'Parameter $r_{x2}$')
ax.set_title(r'Phase Diagram: Varying $r_{x1}, r_{x2}$')
ax.grid(True, linestyle=':', alpha=0.5)

legend_elements = [
    Patch(facecolor='gold', edgecolor='k', label='Coexistence'),
    Patch(facecolor='skyblue', edgecolor='k', label='$x_1 < 0$ (Constant)'),
    Patch(facecolor='salmon', edgecolor='k', label='$x_2 < 0$ (Constant)'),
    Patch(facecolor='SlateBlue', edgecolor='k', label='$y_1 < 0$'),
    Patch(facecolor='IndianRed', edgecolor='k', label='$y_2 < 0$'),
    
    Line2D([0], [0], color='SlateBlue', lw=2, linestyle='-', label=r'$y_1=0$ Boundary'),
    Line2D([0], [0], color='IndianRed', lw=2, linestyle='-', label=r'$y_2=0$ Boundary'),
]

ax.legend(handles=legend_elements, loc='lower right')
plt.savefig("r_x1vsr_x2")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# 1. パラメータ設定
# ry1, ry2 を変数とし、rx (およびその他) を固定します
params = {
    'rx': [1.5, 1.0],               # [rx1, rx2] (固定)
    'lambda_diag': [1.0, 1.0],      # [l11, l22]
    'lambda_offdiag': [0.3, 0.3],   # [l12, l21]
    'c': [0.8, 0.6],                # [c1, c2]
    'd': [0.6, 0.8]                 # [d1, d2]
}

rx1, rx2 = params['rx']
l11, l22 = params['lambda_diag']
l12, l21 = params['lambda_offdiag']
c1, c2 = params['c']
d1, d2 = params['d']

# 2. メッシュグリッド (ry1, ry2 を変数とする)
res = 400
range_ry = 2.0
vals = np.linspace(0.001, range_ry, res)
RY1, RY2 = np.meshgrid(vals, vals)

# 3. 計算
# --- Det X & Det Y (すべて定数) ---
Det_x = l11 * l22 - l12 * l21
Det_y = c1 * d2 * l11 * l22 - c2 * d1 * l12 * l21

# --- Num Y1 & Y2 (すべて定数) ---
# Yの分子は rx に依存し、ry には依存しない
Num_y1 = d2 * l22 * rx1 - d1 * l21 * rx2
Num_y2 = c1 * l11 * rx2 - c2 * l12 * rx1

# --- Num X1 & X2 (変数 RY1, RY2 に依存) ---
# Num_x1 = l22 * ry1 - l12 * ry2
Num_x1 = l22 * RY1 - l12 * RY2

# Num_x2 = l11 * ry2 - l21 * ry1
Num_x2 = l11 * RY2 - l21 * RY1

# 平衡点の値を計算
with np.errstate(divide='ignore', invalid='ignore'):
    # Det定数、Num_y定数 -> Yはスカラー（全領域で同じ値）
    Y1 = Num_y1 / Det_y
    Y2 = Num_y2 / Det_y
    
    # Xはグリッド
    X1 = Num_x1 / Det_x
    X2 = Num_x2 / Det_x

# 4. 領域のカテゴリ分け
# Y1, Y2 はスカラーですが、判定用にそのまま使えます
Category = np.zeros_like(X1, dtype=int)

pos_x1 = X1 > 0
pos_x2 = X2 > 0
# Yについてはスカラー判定
is_y1_pos = Y1 > 0
is_y2_pos = Y2 > 0

# カテゴリ番号定義:
# 1: 共存, 2: x1<0, 3: x2<0, 4: y1<0, 5: y2<0

# Yの死滅判定 (Yが生きていないと、全画面がその色になります)
if not is_y1_pos:
    Category[:] = 4
elif not is_y2_pos: # elifなのでy1優先
    Category[:] = 5
else:
    # Yが生きている場合のみ、Xの変化を描画
    
    # Xの死滅判定
    mask_x1_die = (~pos_x1) & pos_x2
    Category[mask_x1_die] = 2

    mask_x2_die = pos_x1 & (~pos_x2)
    Category[mask_x2_die] = 3
    
    # 両方Xが死ぬエリア（厳密には定義外ですが便宜上x1死にしておくか、別途定義）
    mask_both_die = (~pos_x1) & (~pos_x2)
    # ここでは便宜上、より条件が厳しい方、あるいは定義順に従い処理
    # 例として「両方死」は、生物学的にはありえないパラメータ領域（両方マイナス）
    
    # 共存領域
    mask_coexist = pos_x1 & pos_x2
    Category[mask_coexist] = 1

# 5. プロット
fig, ax = plt.subplots(figsize=(10, 8))

# カラーマップ
colors = ['white', 'gold', 'skyblue', 'salmon', 'SlateBlue', 'IndianRed']
cmap = ListedColormap(colors)
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5], cmap.N)

im = ax.imshow(Category, extent=[0, range_ry, 0, range_ry], origin='lower',
               cmap=cmap, norm=norm, aspect='auto', alpha=0.9)

# --- 曲線 (境界線) 描画 ---
x_line = np.linspace(0.001, range_ry, 500)

# 1. Num_x1 = 0 (x1の符号変化)
# l22 * ry1 = l12 * ry2
# ry2 = (l22 / l12) * ry1
slope_x1 = l22 / l12
y_line_x1 = slope_x1 * x_line
ax.plot(x_line, y_line_x1, color='skyblue', linestyle='-', linewidth=2.5, label=r'$x_1=0$ Boundary')

# 2. Num_x2 = 0 (x2の符号変化)
# l11 * ry2 = l21 * ry1
# ry2 = (l21 / l11) * ry1
slope_x2 = l21 / l11
y_line_x2 = slope_x2 * x_line
ax.plot(x_line, y_line_x2, color='salmon', linestyle='-', linewidth=2.5, label=r'$x_2=0$ Boundary')

# --- パラメータ表示 ---
textstr = '\n'.join(( 
    r'Fixed Parameters:',
    r'$r_x = [%.1f, %.1f]$' % tuple(params['rx']),
    r'$\lambda = [%.1f, %.1f, %.1f, %.1f]$' % (l11, l12, l21, l22),
    r'$c = [%.1f, %.1f]$' % (c1, c2),
    r'$d = [%.1f, %.1f]$' % (d1, d2)
))

props = dict(boxstyle='round', facecolor='white', alpha=0.9)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=props)

# 軸ラベルとタイトル
ax.set_xlim(0, range_ry)
ax.set_ylim(0, range_ry)
ax.set_xlabel(r'Parameter $r_{y1}$')
ax.set_ylabel(r'Parameter $r_{y2}$')
ax.set_title(r'Phase Diagram: Varying $r_{y1}, r_{y2}$')
ax.grid(True, linestyle=':', alpha=0.5)

legend_elements = [
    Patch(facecolor='gold', edgecolor='k', label='Coexistence'),
    Patch(facecolor='skyblue', edgecolor='k', label='$x_1 < 0$'),
    Patch(facecolor='salmon', edgecolor='k', label='$x_2 < 0$'),
    Patch(facecolor='SlateBlue', edgecolor='k', label='$y_1 < 0$ (Constant)'),
    Patch(facecolor='IndianRed', edgecolor='k', label='$y_2 < 0$ (Constant)'),
    
    Line2D([0], [0], color='skyblue', lw=2.5, linestyle='-', label=r'$x_1=0$ Boundary'),
    Line2D([0], [0], color='salmon', lw=2.5, linestyle='-', label=r'$x_2=0$ Boundary'),
]

ax.legend(handles=legend_elements, loc='upper right') # 線が右下に伸びるので左上に配置

plt.savefig("r_y1vsr_y2")

plt.show()